# Keras Functional API: Multi-Input/Output Models, Shared Layers, and Branching Architectures

> This notebook contains all code examples from the blog post.
> [Read the full post on BotMartz](https://blog.botmartz.com/tensorflow_5)

**Author:** Soham Sharma · blog.botmartz.com

In [ ]:
!pip install -q tensorflow mlflow

In [ ]:
import tensorflow as tf

# Input: symbolic tensor representing the data
inputs = tf.keras.Input(shape=(784,), name="input_layer")
print(f"Type: {type(inputs)}")
print(f"Shape: {inputs.shape}")
print(f"Name: {inputs.name}")

# Each layer call returns a new symbolic tensor
x = tf.keras.layers.Dense(256, activation='relu')(inputs)
print(f"\nAfter Dense(256): {x.shape}")

outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
print(f"After Dense(10):  {outputs.shape}")

# Model is defined by its terminal inputs and outputs
model = tf.keras.Model(inputs=inputs, outputs=outputs)
print(f"\nModel input:  {model.input_shape}")
print(f"Model output: {model.output_shape}")

In [ ]:
import tensorflow as tf
import numpy as np

# Shared encoder
inputs = tf.keras.Input(shape=(128,), name="features")
x = tf.keras.layers.Dense(256, activation='relu', name="shared_1")(inputs)
x = tf.keras.layers.BatchNormalization(name="shared_bn")(x)
x = tf.keras.layers.Dense(128, activation='relu', name="shared_2")(x)
shared_repr = tf.keras.layers.Dropout(0.3, name="shared_dropout")(x)

# Task 1: binary sentiment classification
sentiment_x = tf.keras.layers.Dense(64, activation='relu', name="sentiment_dense")(shared_repr)
sentiment_out = tf.keras.layers.Dense(1, activation='sigmoid', name="sentiment")(sentiment_x)

# Task 2: topic classification (5 classes)
topic_x = tf.keras.layers.Dense(64, activation='relu', name="topic_dense")(shared_repr)
topic_out = tf.keras.layers.Dense(5, activation='softmax', name="topic")(topic_x)

# Task 3: urgency regression (predict a score 0-1)
urgency_x = tf.keras.layers.Dense(32, activation='relu', name="urgency_dense")(shared_repr)
urgency_out = tf.keras.layers.Dense(1, activation='sigmoid', name="urgency")(urgency_x)

model = tf.keras.Model(
    inputs=inputs,
    outputs={"sentiment": sentiment_out, "topic": topic_out, "urgency": urgency_out},
    name="multi_task_model"
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss={
        "sentiment": "binary_crossentropy",
        "topic":     "sparse_categorical_crossentropy",
        "urgency":   "mse",
    },
    loss_weights={"sentiment": 1.0, "topic": 1.0, "urgency": 0.5},
    metrics={
        "sentiment": ["accuracy"],
        "topic":     ["accuracy"],
    },
)

total_params = model.count_params()
print(f"Total parameters: {total_params:,}")
print(f"Output names: {list(model.output_names)}")

In [ ]:
import numpy as np

np.random.seed(42)
N = 2000
X = np.random.randn(N, 128).astype(np.float32)
y_sentiment = np.random.randint(0, 2, N).astype(np.float32)
y_topic = np.random.randint(0, 5, N).astype(np.int32)
y_urgency = np.random.rand(N).astype(np.float32)

history = model.fit(
    X,
    {"sentiment": y_sentiment, "topic": y_topic, "urgency": y_urgency},
    epochs=3,
    batch_size=64,
    validation_split=0.2,
    verbose=1,
)

# Inference on new data
x_new = np.random.randn(4, 128).astype(np.float32)
predictions = model.predict(x_new, verbose=0)
print(f"\nInference results:")
print(f"  Sentiment (binary): {predictions['sentiment'].flatten().round(3)}")
print(f"  Topic (5-class): {predictions['topic'].argmax(axis=1)}")
print(f"  Urgency (0-1): {predictions['urgency'].flatten().round(3)}")

In [ ]:
import tensorflow as tf
import numpy as np

def build_encoder(input_dim: int, embedding_dim: int = 64) -> tf.keras.Model:
    """Shared encoder used by both branches."""
    inputs = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(128, activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(embedding_dim)(inputs)
    # L2-normalize the embedding
    outputs = tf.keras.layers.Lambda(
        lambda t: tf.math.l2_normalize(t, axis=1), name="l2_norm"
    )(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name="encoder")

# Shared encoder — same layer object, same weights
encoder = build_encoder(input_dim=50, embedding_dim=32)

# Two inputs — same shape
input_a = tf.keras.Input(shape=(50,), name="input_a")
input_b = tf.keras.Input(shape=(50,), name="input_b")

# Both inputs pass through the SAME encoder (shared weights)
embedding_a = encoder(input_a)
embedding_b = encoder(input_b)

# Cosine similarity between the two embeddings
cosine_sim = tf.keras.layers.Dot(axes=1, normalize=False, name="cosine_sim")(
    [embedding_a, embedding_b]
)

# Contrastive output: high similarity → same class
similarity_output = tf.keras.layers.Activation('sigmoid', name="similarity")(cosine_sim)

siamese = tf.keras.Model(
    inputs={"input_a": input_a, "input_b": input_b},
    outputs=similarity_output,
    name="siamese_network"
)

# Verify weight sharing: both branches use the same encoder
print(f"Total model params:    {siamese.count_params():,}")
print(f"Encoder params:        {encoder.count_params():,}")
print(f"Non-encoder params:    {siamese.count_params() - encoder.count_params():,}")
print(f"\nWeight sharing verified: {id(siamese.get_layer('encoder')) == id(siamese.get_layer('encoder'))}")

In [ ]:
import numpy as np
import tensorflow as tf

siamese.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# Generate pairs: positive (same class, label=1) and negative (different class, label=0)
np.random.seed(42)
N = 1000
X_a = np.random.randn(N, 50).astype(np.float32)
X_b = np.random.randn(N, 50).astype(np.float32)
# Half pairs are "similar" (random assignment for demo)
labels = np.random.randint(0, 2, N).astype(np.float32)

history = siamese.fit(
    {"input_a": X_a, "input_b": X_b},
    labels,
    epochs=3,
    batch_size=64,
    validation_split=0.2,
    verbose=0,
)

final_val_acc = history.history['val_accuracy'][-1]
print(f"Final val accuracy: {final_val_acc:.4f}")

# Inference
new_a = np.random.randn(3, 50).astype(np.float32)
new_b = np.random.randn(3, 50).astype(np.float32)
similarities = siamese.predict({"input_a": new_a, "input_b": new_b}, verbose=0)
print(f"Similarity scores: {similarities.flatten().round(3)}")

In [ ]:
import tensorflow as tf
import numpy as np

def build_multiscale_encoder(seq_len: int = 64, d: int = 32) -> tf.keras.Model:
    inputs = tf.keras.Input(shape=(seq_len, d), name="sequence")

    # Branch 1: local patterns (small receptive field)
    x1 = tf.keras.layers.Conv1D(64, kernel_size=3, padding='same', activation='relu', name="local")(inputs)
    x1 = tf.keras.layers.GlobalAveragePooling1D(name="local_gap")(x1)

    # Branch 2: medium-range patterns
    x2 = tf.keras.layers.Conv1D(64, kernel_size=8, padding='same', activation='relu', name="medium")(inputs)
    x2 = tf.keras.layers.GlobalAveragePooling1D(name="medium_gap")(x2)

    # Branch 3: global patterns (looks at full sequence)
    x3 = tf.keras.layers.GlobalAveragePooling1D(name="global_gap")(inputs)
    x3 = tf.keras.layers.Dense(64, activation='relu', name="global_dense")(x3)

    # Concatenate all scales
    merged = tf.keras.layers.Concatenate(name="merge")([x1, x2, x3])
    merged = tf.keras.layers.Dense(128, activation='relu', name="combined")(merged)
    merged = tf.keras.layers.Dropout(0.3)(merged)
    outputs = tf.keras.layers.Dense(5, activation='softmax', name="class_output")(merged)

    return tf.keras.Model(inputs=inputs, outputs=outputs, name="multiscale_encoder")

model = build_multiscale_encoder()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

np.random.seed(0)
X = np.random.randn(500, 64, 32).astype(np.float32)
y = np.random.randint(0, 5, 500).astype(np.int32)

history = model.fit(X, y, epochs=3, batch_size=32, validation_split=0.2, verbose=0)

print(f"Model params: {model.count_params():,}")
print(f"Final val accuracy: {history.history['val_accuracy'][-1]:.4f}")

# Verify output
pred = model.predict(X[:4], verbose=0)
print(f"Prediction shape: {pred.shape}, sum per row: {pred.sum(axis=1).round(3)}")

In [ ]:
import tensorflow as tf

# Build a model with named sub-components
inputs = tf.keras.Input(shape=(100,))
frozen_branch = tf.keras.layers.Dense(64, activation='relu', name="frozen_dense")(inputs)
trainable_branch = tf.keras.layers.Dense(64, activation='relu', name="trainable_dense")(inputs)
merged = tf.keras.layers.Concatenate()([frozen_branch, trainable_branch])
output = tf.keras.layers.Dense(5, activation='softmax')(merged)

model = tf.keras.Model(inputs, output)

# Freeze the frozen branch
model.get_layer("frozen_dense").trainable = False

# Count trainable vs total
total = model.count_params()
trainable = sum(tf.size(p).numpy() for p in model.trainable_variables)
non_trainable = sum(tf.size(p).numpy() for p in model.non_trainable_variables)

print(f"Total params:         {total:,}")
print(f"Trainable params:     {trainable:,}")
print(f"Non-trainable params: {non_trainable:,}")